## Simple Agent with Microsoft Agent Framework

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.8

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Creating the Foundry AI Project Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential

foundry_project_client = AIProjectClient(
    endpoint = foundry_project_endpoint,
    credential = credential
)

### Creating the Agent

In [ ]:
agent = Agent(
    client = FoundryChatClient(
        project_endpoint = foundry_project_endpoint,
        model = model_deployment_name,
        credential = credential
    ),
    name = "Batman-Agent",
    instructions = "You are Batman, the Protector and Dark Knight of Gotham City",
)

### Call the Agent Locally

In [ ]:
result = await agent.run("Hi, Kuljot here!.....How is Gotham City Doing today?")
print(f"Agent: {result}")

### Create and Use the Agent in a Server-Side Fashion

In [ ]:
server_side_agent = foundry_project_client.agents.create_version(
    agent_name = agent.name,
    definition = to_prompt_agent(agent)
)

print(f"Published prompt agent: {server_side_agent.name} v{server_side_agent.version}\n")

In [ ]:
foundry_agent = FoundryAgent(
    project_endpoint=foundry_project_endpoint,
    agent_name=server_side_agent.name,
    agent_version=server_side_agent.version,
    credential=credential
)

### Create a session to maintain conversation history

In [ ]:
session = foundry_agent.create_session()

### Request a non-streaming response from the Agent

In [ ]:
result = await foundry_agent.run("Hi, Kuljot here!.....How is Gotham City Doing today?", session=session)
print(f"Agent: {result}")

### Request streaming responses from the Agent

In [ ]:
print("Agent (streaming): ", end="", flush=True)
async for chunk in foundry_agent.run("do you remember any info about me?", session=session, stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
print()